# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [5]:
import pandas as pd
import geopandas as gpd
import os

In [43]:
def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

In [17]:
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\subdivisão bacias bis\inputs_SIG\4216206.csv',delimiter = ';')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\subdivisão bacias bis\bacias_02.10\EnviarGabriel.gpkg')
economias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\subdivisão bacias bis\inputs_SIG\Consumidores SFS dez 2024 shp\Consumidores_SFS_dez_2024.shp',encoding='latin1')
coluna_nome_bacias = 'Text'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\subdivisão bacias bis\arquivo_08_10.xlsx'
crs = "EPSG:31982"

In [57]:
economias['ECO_RESIDE'] = pd.to_numeric(economias['ECO_RESIDE'], errors='coerce')
economias_reside = economias[economias['ECO_RESIDE'] != 0]

In [41]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1) ]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

In [51]:
domparticular = domparticular.to_crs(crs)
economias_reside = economias_reside.to_crs(crs)

# defina o CRS correto de ORIGEM do bacias (sem reprojetar ainda)
bacias = bacias.set_crs("EPSG:31982", allow_override=True)

# agora reprojete para o CRS-alvo que você está usando no resto do fluxo
bacias = bacias.to_crs(crs)

In [77]:
dom_bacias = contar_pontos_poligono(bacias, domparticular)
lig_bacias = contar_pontos_poligono(bacias, economias_reside)


In [116]:
eco_bacias = gpd.sjoin(
    economias_reside,
    bacias,  
    predicate="intersects",
    how="left"
)


In [118]:
eco_bacias_imp = eco_bacias[['ECO_RESIDE','Text']].copy()

In [134]:
eco_bacias = eco_bacias_imp.groupby("Text").sum().reset_index()


,Text,ECO_RESIDE
0,BT-01A,80
1,BT-01A.1,12
2,BT-01B,294
3,BT-01B.1,4
4,BT-01B.2,61
5,BT-01B.3,22
6,BT-01B.4,7
7,BT-01C,108
8,BT-01C.1,6
9,BT-02A,96


In [96]:
dom_bacias = dom_bacias.rename(columns={'n_pontos': 'num_domicilios'})

In [106]:
lig_bacias = lig_bacias.rename(columns={'n_pontos': 'num_ligacoes'})

In [138]:
eco_bacias = eco_bacias.rename(columns={'ECO_RESIDE': 'num_economias'})

In [144]:
lig_bacias[['Text','num_ligacoes']]
dom_bacias[['Text','num_domicilios']]
eco_bacias[['Text','num_economias']]

,Text,num_ligacoes
0,OL01A,97
1,BT-01A,79
2,BT-01B,266
3,BT-01B.1,4
4,BT-01B.2,58
5,BT-01B.3,22
6,BT-01B.4,7
7,BT-01C,106
8,BT-01C.1,6
9,BT-02A.2,118


In [146]:
# Mesclar lig_bacias e dom_bacias pela coluna "Text"
df_merge = pd.merge(
    lig_bacias[['Text', 'num_ligacoes']],
    dom_bacias[['Text', 'num_domicilios']],
    on='Text',
    how='outer'   # ou 'inner' se quiser só os textos que existem em ambas
)

# Agora mesclar eco_bacias também
df_merge = pd.merge(
    df_merge,
    eco_bacias[['Text', 'num_economias']],
    on='Text',
    how='outer'
)

# Resultado final
df_merge.head()

,Text,num_ligacoes,num_domicilios,num_economias
0,OL01A,97,131,107
1,BT-01A,79,87,80
2,BT-01B,266,264,294
3,BT-01B.1,4,8,4
4,BT-01B.2,58,94,61


In [150]:
df_merge.to_excel(r'C:\Users\gabriel.coimbra\Desktop\subdivisão bacias bis\bacias_contabilizadas.xlsx')